In [1]:
!pip install librosa lightgbm optuna pyloudnorm --quiet

In [2]:
import os
import subprocess
import warnings
import kagglehub
import pickle
import numpy as np
import librosa
import pyloudnorm as pyln
import scipy.signal as sig
from scipy.stats import kurtosis, skew
from collections import Counter

warnings.filterwarnings('ignore')

In [3]:
SAMPLE_RATE          = 16000
CLIP_DURATION        = 3
TARGET_LEN           = SAMPLE_RATE * CLIP_DURATION
N_MFCC               = 13
N_MELS               = 40
N_MFCC_QUARTERS      = 4
TARGET_LUFS          = -23.0
STEP_SIZE            = 0.25
RESP_LOW_HZ          = 80
RESP_HIGH_HZ         = 2500
SILENCE_THRESHOLD    = 0.001
CONFIDENCE_HIGH      = 0.75
CONFIDENCE_LOW       = 0.55
MIN_AUDIO_SECS       = 10.0
MIN_WINDOWS_RELIABLE = 5

In [4]:
print(f'Step size          : {STEP_SIZE}s')
print(f'Min audio for trust: {MIN_AUDIO_SECS}s')
print(f'Min windows trusted: {MIN_WINDOWS_RELIABLE}')
overlap_pct = (1 - STEP_SIZE / CLIP_DURATION) * 100
print(f'Window overlap     : {overlap_pct:.0f}% '
      f'(consecutive windows share {overlap_pct:.0f}% of the same audio)')

Step size          : 0.25s
Min audio for trust: 10.0s
Min windows trusted: 5
Window overlap     : 92% (consecutive windows share 92% of the same audio)


In [5]:
path = kagglehub.dataset_download("shafayatulislam/real-audio-data")
print("Dataset path:", path)

Dataset path: /kaggle/input/datasets/shafayatulislam/real-audio-data


In [6]:
ORIG_AUDIO_PATH = "/kaggle/input/datasets/shafayatulislam/real-audio-data/yo.m4a"
WAV_AUDIO_PATH  = "/kaggle/working/test_audio.wav"

In [7]:
result = subprocess.run(
    ["ffmpeg", "-y", "-i", ORIG_AUDIO_PATH,
     "-ar", str(SAMPLE_RATE), "-ac", "1", WAV_AUDIO_PATH],
    capture_output=True, text=True
)

In [8]:
if result.returncode != 0:
    print("ffmpeg conversion failed:", result.stderr[-500:])
    TEST_AUDIO_PATH = ORIG_AUDIO_PATH
else:
    TEST_AUDIO_PATH = WAV_AUDIO_PATH

full_audio, _ = librosa.load(TEST_AUDIO_PATH, sr=SAMPLE_RATE, mono=True)
audio_duration_secs = len(full_audio) / SAMPLE_RATE
print(f"Loaded : {len(full_audio)} samples  ({audio_duration_secs:.2f}s total)")
print(f"RMS (raw): {np.sqrt(np.mean(full_audio**2)):.4f}")

Loaded : 237909 samples  (14.87s total)
RMS (raw): 0.0793


In [9]:
duration_ok = audio_duration_secs >= MIN_AUDIO_SECS

if not duration_ok:
    n_expected_wins = max(1, int((audio_duration_secs - CLIP_DURATION) / STEP_SIZE) + 1)
    print()
    print(f"[WARNING] Recording is only {audio_duration_secs:.1f}s "
          f"(minimum: {MIN_AUDIO_SECS}s).")
    print(f"  Only ~{n_expected_wins} window(s) will be produced — "
          f"and with {(1 - STEP_SIZE/CLIP_DURATION)*100:.0f}% overlap they are NOT independent.")
else:
    print(f"Sufficient ({audio_duration_secs:.1f}s).")

Sufficient (14.9s).


In [10]:
def apply_bandpass(audio: np.ndarray,
                   sr: int = SAMPLE_RATE,
                   low_hz: float = RESP_LOW_HZ,
                   high_hz: float = RESP_HIGH_HZ) -> np.ndarray:
    nyq  = sr / 2.0
    low  = max(low_hz  / nyq, 0.001)
    high = min(high_hz / nyq, 0.999)
    b, a = sig.butter(4, [low, high], btype='band')
    return sig.filtfilt(b, a, audio).astype(np.float32)

In [11]:
def spectral_subtraction(audio: np.ndarray,
                         sr: int = SAMPLE_RATE,
                         noise_frames: int = 20) -> np.ndarray:
    n_fft      = 512
    hop_length = 128

    stft      = librosa.stft(audio, n_fft=n_fft, hop_length=hop_length)
    magnitude = np.abs(stft)
    phase     = np.angle(stft)

    frame_energies = np.sum(magnitude ** 2, axis=0)
    quiet_idx      = np.argsort(frame_energies)[:noise_frames]
    noise_estimate = np.mean(magnitude[:, quiet_idx], axis=1, keepdims=True)

    alpha = 2.0
    beta  = 0.01
    magnitude_clean = np.maximum(
        magnitude - alpha * noise_estimate,
        beta * magnitude
    )

    stft_clean  = magnitude_clean * np.exp(1j * phase)
    audio_clean = librosa.istft(stft_clean, hop_length=hop_length, length=len(audio))
    return audio_clean.astype(np.float32)

In [12]:
def preprocess_phone_audio(audio: np.ndarray, sr: int = SAMPLE_RATE) -> np.ndarray:
    print("[1/3] Bandpass filter (80–2500 Hz)…")
    audio = apply_bandpass(audio, sr)

    print("[2/3] Spectral subtraction (noise reduction)…")
    audio = spectral_subtraction(audio, sr)

    print("[3/3] Soft clip…")
    audio = np.clip(audio, -1.0, 1.0).astype(np.float32)

    return audio

In [13]:
rms_before = np.sqrt(np.mean(full_audio ** 2))
full_audio_clean = preprocess_phone_audio(full_audio)
rms_after = np.sqrt(np.mean(full_audio_clean ** 2))
print(f"\nRMS before: {rms_before:.4f}")
print(f"RMS after : {rms_after:.4f}")

[1/3] Bandpass filter (80–2500 Hz)…
[2/3] Spectral subtraction (noise reduction)…
[3/3] Soft clip…

RMS before: 0.0793
RMS after : 0.0688


In [14]:
def normalize_loudness(audio: np.ndarray,
                       sr: int = SAMPLE_RATE,
                       target_lufs: float = TARGET_LUFS) -> np.ndarray:
    meter    = pyln.Meter(sr)
    audio64  = audio.astype(np.float64)
    loudness = meter.integrated_loudness(audio64)
    if not (np.isinf(loudness) or np.isnan(loudness)):
        audio64 = pyln.normalize.loudness(audio64, loudness, target_lufs)
    else:
        print("[warn] Loudness measurement returned inf/nan")
    return np.clip(audio64, -1.0, 1.0).astype(np.float32)

In [15]:
full_audio_clean = normalize_loudness(full_audio_clean)
print(f"Loudness normalised to {TARGET_LUFS} LUFS")
print(f"Final RMS: {np.sqrt(np.mean(full_audio_clean**2)):.4f}")

Loudness normalised to -23.0 LUFS
Final RMS: 0.0561


In [16]:
def extract_features(audio: np.ndarray, sr: int = SAMPLE_RATE) -> np.ndarray:
    feats = []
    mfcc    = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=N_MFCC)
    d_mfcc  = librosa.feature.delta(mfcc)
    d2_mfcc = librosa.feature.delta(mfcc, order=2)

    n_frames = mfcc.shape[1]
    q_size   = max(1, n_frames // N_MFCC_QUARTERS)

    for matrix in (mfcc, d_mfcc, d2_mfcc):
        for q in range(N_MFCC_QUARTERS):
            seg = matrix[:, q * q_size : (q + 1) * q_size]
            if seg.shape[1] == 0:
                seg = matrix[:, -1:]
            feats += list(np.mean(seg, axis=1))
            feats += list(np.std(seg,  axis=1))

    feats += list(kurtosis(mfcc, axis=1, nan_policy='omit'))
    feats += list(skew(    mfcc, axis=1, nan_policy='omit'))

    mel    = librosa.feature.melspectrogram(y=audio, sr=sr, n_mels=N_MELS)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    feats += list(np.mean(mel_db, axis=1))
    feats += list(np.std( mel_db, axis=1))

    contrast = librosa.feature.spectral_contrast(y=audio, sr=sr, n_bands=6)
    feats += list(np.mean(contrast, axis=1))
    feats += list(np.std( contrast, axis=1))

    chroma = librosa.feature.chroma_stft(y=audio, sr=sr)
    feats += list(np.mean(chroma, axis=1))
    feats += list(np.std( chroma, axis=1))

    for feat_fn in (
        lambda: librosa.feature.zero_crossing_rate(y=audio),
        lambda: librosa.feature.spectral_centroid(y=audio, sr=sr),
        lambda: librosa.feature.spectral_rolloff(y=audio,  sr=sr),
        lambda: librosa.feature.spectral_bandwidth(y=audio, sr=sr),
        lambda: librosa.feature.rms(y=audio),
    ):
        v = feat_fn()
        feats += [float(np.mean(v)), float(np.std(v))]

    return np.array(feats, dtype=np.float32)

In [17]:
path = kagglehub.dataset_download("shafayatulislam/trainedmodels")
print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/datasets/shafayatulislam/trainedmodels


In [18]:
def load_model(base_name: str, versions: list = ['v4', 'v3']):
    base = '/kaggle/input/datasets/shafayatulislam/trainedmodels'
    for ver in versions:
        path = os.path.join(base, f'{base_name}_{ver}.pkl')
        if os.path.exists(path):
            with open(path, 'rb') as f:
                obj = pickle.load(f)
            print(f"  Loaded {os.path.basename(path)}")
            return obj
    raise FileNotFoundError(f"No model found for {base_name} in {versions}")

scaler   = load_model('scaler')
encoder  = load_model('encoder')
ensemble = load_model('ensemble')
print("\nAll models loaded. Classes:", list(encoder.classes_))

  Loaded scaler_v4.pkl
  Loaded encoder_v4.pkl
  Loaded ensemble_v4.pkl

All models loaded. Classes: [np.str_('crackle'), np.str_('normal'), np.str_('snore'), np.str_('wheeze')]


In [19]:
def predict_sliding_windows(audio: np.ndarray,
                             sr:    int   = SAMPLE_RATE,
                             clip_len: int = TARGET_LEN,
                             step_sec: float = STEP_SIZE,
                             silence_thr: float = SILENCE_THRESHOLD) -> list:
    step    = int(step_sec * sr)
    results = []

    if len(audio) < clip_len:
        pad_secs = (clip_len - len(audio)) / sr
        print(f"[INFO] Audio shorter than clip length — zero-padding by {pad_secs:.2f}s")
        audio = np.pad(audio, (0, clip_len - len(audio)))

    starts = list(range(0, len(audio) - clip_len + 1, step))
    n_skipped_silent = 0

    for idx, start in enumerate(starts):
        window = audio[start : start + clip_len].copy()

        rms = float(np.sqrt(np.mean(window ** 2)))
        if rms < silence_thr:
            n_skipped_silent += 1
            print(f"  Win {idx+1:>2} ({start/sr:.2f}s–{(start+clip_len)/sr:.2f}s): "
                  f"SILENT (RMS={rms:.5f}) — skipped")
            continue

        window = normalize_loudness(window, sr)

        feats        = extract_features(window, sr)
        feats_scaled = scaler.transform(feats.reshape(1, -1))
        probs        = ensemble.predict_proba(feats_scaled)[0]

        results.append({
            'window_idx': idx + 1,
            'start_sec' : start / sr,
            'end_sec'   : (start + clip_len) / sr,
            'rms'       : rms,
            'probs'     : probs,
        })

    if n_skipped_silent:
        print(f"\n[INFO] {n_skipped_silent} silent window(s) skipped.")
    return results


print(f"Running sliding-window prediction "
      f"(window={CLIP_DURATION}s, step={STEP_SIZE}s)…\n")
window_results = predict_sliding_windows(full_audio_clean)
print(f"Total windows analysed : {len(window_results)}")

overlap_pct = (1 - STEP_SIZE / CLIP_DURATION) * 100
if len(window_results) > 1 and not duration_ok:
    print(f"[INFO] Window overlap is {overlap_pct:.0f}% — "
          f"windows share most of the same audio and are NOT independent.")

if len(window_results) < MIN_WINDOWS_RELIABLE:
    print(f"[WARNING] Only {len(window_results)} window(s) analysed "
          f"(need >= {MIN_WINDOWS_RELIABLE} for a reliable prediction).")
    print("Final result should be treated as indicative only.")

Running sliding-window prediction (window=3s, step=0.25s)…

Total windows analysed : 48


In [20]:
for r in window_results:
    row = (f"{r['window_idx']:>3}  "
           f"{r['start_sec']:>5.2f}s–{r['end_sec']:>5.2f}s  ")
    for p in r['probs']:
        row += f"{p*100:>8.1f}%"
    predicted = encoder.classes_[np.argmax(r['probs'])]
    conf      = float(np.max(r['probs']))
    if conf >= CONFIDENCE_HIGH:
        flag = "✓"
    elif conf >= CONFIDENCE_LOW:
        flag = "~"
    else:
        flag = "?"
    row += f"  → {predicted.capitalize()} {conf*100:.1f}% {flag}"
    print(row)

  1   0.00s– 3.00s      12.1%    45.8%     1.9%    40.3%  → Normal 45.8% ?
  2   0.25s– 3.25s      11.5%    57.4%     2.2%    28.9%  → Normal 57.4% ~
  3   0.50s– 3.50s      11.9%    56.0%     2.5%    29.6%  → Normal 56.0% ~
  4   0.75s– 3.75s      10.7%    55.7%     2.1%    31.4%  → Normal 55.7% ~
  5   1.00s– 4.00s      12.0%    56.7%     2.4%    28.9%  → Normal 56.7% ~
  6   1.25s– 4.25s      11.6%    57.6%     1.8%    29.0%  → Normal 57.6% ~
  7   1.50s– 4.50s      11.1%    56.2%     1.9%    30.7%  → Normal 56.2% ~
  8   1.75s– 4.75s      10.9%    55.3%     2.1%    31.7%  → Normal 55.3% ~
  9   2.00s– 5.00s      11.1%    51.5%     2.2%    35.2%  → Normal 51.5% ?
 10   2.25s– 5.25s      11.8%    49.5%     2.2%    36.5%  → Normal 49.5% ?
 11   2.50s– 5.50s      12.3%    45.3%     2.3%    40.1%  → Normal 45.3% ?
 12   2.75s– 5.75s      12.0%    48.1%     2.3%    37.6%  → Normal 48.1% ?
 13   3.00s– 6.00s      11.5%    46.1%     2.5%    39.9%  → Normal 46.1% ?
 14   3.25s– 6.25s      1

In [21]:
if not window_results:
    print("ERROR: No valid windows found. Check the audio file and recording length.")
    aggregated    = None
    pred_label    = None
    max_conf      = 0.0
    low_data      = True
    low_data_reasons = ["no valid windows found"]
else:
    all_probs   = np.array([r['probs'] for r in window_results])
    rms_weights = np.array([r['rms']   for r in window_results])
    rms_weights = rms_weights / rms_weights.sum()

    aggregated = np.average(all_probs, axis=0, weights=rms_weights)

    per_window_winners = [encoder.classes_[np.argmax(p)] for p in all_probs]
    vote_counts  = Counter(per_window_winners)
    vote_winner  = vote_counts.most_common(1)[0][0]
    vote_pct     = vote_counts.most_common(1)[0][1] / len(per_window_winners) * 100

    print("Weighted-average probabilities")
    for i, cls in enumerate(encoder.classes_):
        print(f"  {cls.capitalize():<8} : {aggregated[i]*100:>5.1f}%")

    print("\nPer-window vote counts")
    for cls, cnt in vote_counts.most_common():
        print(f"  {cls.capitalize():<8} : {cnt} window(s) ({cnt/len(per_window_winners)*100:.0f}%)")

    weighted_winner = encoder.classes_[np.argmax(aggregated)]
    methods_agree   = (weighted_winner == vote_winner)
    print(f"\nMethods agree: {'YES' if methods_agree else 'NO'}")

    max_idx    = int(np.argmax(aggregated))
    max_conf   = float(aggregated[max_idx])
    pred_label = encoder.classes_[max_idx]

    low_data_reasons = []
    if len(window_results) < MIN_WINDOWS_RELIABLE:
        low_data_reasons.append(
            f"only {len(window_results)} active window(s) (need >= {MIN_WINDOWS_RELIABLE})"
        )
    if not duration_ok:       
        low_data_reasons.append(
            f"duration {audio_duration_secs:.1f}s is below the {MIN_AUDIO_SECS}s minimum"
        )
    low_data = bool(low_data_reasons)

Weighted-average probabilities
  Crackle  :  11.6%
  Normal   :  51.5%
  Snore    :   2.1%
  Wheeze   :  34.8%

Per-window vote counts
  Normal   : 42 window(s) (88%)
  Wheeze   : 6 window(s) (12%)

Methods agree: YES


In [22]:
dur_status = 'OK' if duration_ok else f'INSUFFICIENT (min {MIN_AUDIO_SECS}s)'
win_status = 'OK' if len(window_results) >= MIN_WINDOWS_RELIABLE \
             else f'INSUFFICIENT (need >= {MIN_WINDOWS_RELIABLE})'

print(f"Audio duration  : {audio_duration_secs:.2f}s  [{dur_status}]")
print(f"Windows analysed: {len(window_results)}         [{win_status}]")
print(f"Step size used  : {STEP_SIZE}s")
print(f"Window overlap  : {(1 - STEP_SIZE/CLIP_DURATION)*100:.0f}%")

if low_data:
    print()
    for reason in low_data_reasons:
        print(f"  • {reason}")
    print(f"\nRecommendation: re-record at least {MIN_AUDIO_SECS}s of audio and run again.")

Audio duration  : 14.87s  [OK]
Windows analysed: 48         [OK]
Step size used  : 0.25s
Window overlap  : 92%


In [23]:
if aggregated is None:
    print("No result — no valid windows were found.")
elif low_data:
    print("━" * 50)
    print("RESULT: UNRELIABLE (insufficient data)")
    print("━" * 50)
    print(f"  Recording : {audio_duration_secs:.1f}s  |  "
          f"Active windows : {len(window_results)}")
    print()
    print("Reason(s):")
    for reason in low_data_reasons:
        print(f"  • {reason}")
    print()
    print(f"Best guess (do NOT act on this): "
          f"{pred_label.capitalize()} ({max_conf*100:.1f}%)")
    print()
    print("Full distribution:")
    for i, cls in enumerate(encoder.classes_):
        print(f"  {cls.capitalize():<8} {aggregated[i]*100:.1f}%")
    print()
    print(f"Please re-record at least {MIN_AUDIO_SECS}s of audio and run again.")
elif max_conf >= CONFIDENCE_HIGH:
    print("━" * 50)
    print(f"RESULT: DETECTED — {pred_label.capitalize()}  ({max_conf*100:.1f}%)")
    print("━" * 50)
elif max_conf >= CONFIDENCE_LOW:
    print("━" * 50)
    print(f"RESULT: LIKELY — {pred_label.capitalize()}  ({max_conf*100:.1f}%)")
    print("━" * 50)
else:
    top2 = np.argsort(aggregated)[::-1][:2]
    print("━" * 50)
    print("RESULT: UNCERTAIN")
    print("━" * 50)
    for i in top2:
        print(f"  {encoder.classes_[i].capitalize():<8} {aggregated[i]*100:.1f}%")
    print()

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
RESULT: UNCERTAIN
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Normal   51.5%
  Wheeze   34.8%

